# Gemma 4 Myanmar → English Zero-shot S2TT

Gemma 4로 동일한 Myanmar test 777개에 대해 두 작업을 독립 실행합니다.

1. Myanmar speech → Myanmar transcript (ASR, CER 진단)
2. Myanmar speech → English text (direct S2TT, BLEU/chrF++ 평가)

결과는 샘플마다 Google Drive의 JSONL에 즉시 추가되므로 런타임이 끊겨도 재실행하면 이어집니다. Gemma 4의 30초 오디오 제한을 넘는 샘플은 무음에 가까운 지점에서 나누어 처리하고 `num_chunks`에 기록합니다.

In [1]:
# Colab 런타임 유형을 GPU로 설정한 뒤 실행하세요. 설치 후 런타임 재시작은 필요하지 않습니다.
%pip install -q -U "transformers>=5.10.1" accelerate huggingface_hub soundfile pyarrow pandas sacrebleu jiwer tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 158.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 177.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 15.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
cudf-cu13 26.6.0 

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import gc, io, json, os, re, tempfile, time, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import torch
from tqdm.auto import tqdm

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


Mounted at /content/drive
PyTorch: 2.11.0+cu130
CUDA: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## Hugging Face 인증

먼저 모델 페이지에서 Gemma 사용 약관을 승인해야 합니다. Colab Secrets에 `HF_TOKEN`을 저장하는 방법을 우선 사용하며, 없으면 로그인 창이 나타납니다.

In [3]:
from huggingface_hub import login, notebook_login

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
else:
    notebook_login()


In [12]:
# ===== 설정 =====
DATA_DIR = Path('/content/drive/MyDrive/S2ST/data')
JSONL_PATH = Path('/content/drive/MyDrive/S2ST/metadata/test_myanmar_english_translated.jsonl')

# T4에서는 E2B부터 권장. E4B 실험 시 아래 값만 google/gemma-4-E4B-it 로 변경합니다.
MODEL_ID = 'google/gemma-4-E2B-it'
MODEL_SLUG = MODEL_ID.split('/')[-1]
OUTPUT_DIR = Path('/content/drive/MyDrive/S2ST/baselines') / f'{MODEL_SLUG}_mya_eng_s2tt'
PREDICTIONS_PATH = OUTPUT_DIR / 'predictions.jsonl'
SUMMARY_PATH = OUTPUT_DIR / 'summary.csv'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 처음에는 5로 smoke test. 확인 후 None으로 바꾸고 같은 셀부터 다시 실행합니다.
LIMIT = None
RUN_ASR = True
RUN_S2TT = True
MAX_NEW_TOKENS_ASR = 256
MAX_NEW_TOKENS_S2TT = 192
MAX_AUDIO_SECONDS = 29.0

print('model:', MODEL_ID)
print('predictions:', PREDICTIONS_PATH)

model: google/gemma-4-E2B-it
predictions: /content/drive/MyDrive/S2ST/baselines/gemma-4-E2B-it_mya_eng_s2tt/predictions.jsonl


In [13]:
# 메타데이터와 parquet 연결 검증
records = []
with JSONL_PATH.open('r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

assert len(records) == 777, f'예상 777행, 실제 {len(records)}행'
required = {'sample_id', 'parquet_file', 'row_index', 'transcription_my', 'translation_en', 'source'}
missing = required - set(records[0])
assert not missing, f'JSONL 필수 키 누락: {missing}'

parquet_cache = {}
for name in sorted({r['parquet_file'] for r in records}):
    path = DATA_DIR / name
    assert path.exists(), f'파일 없음: {path}'
    parquet_cache[name] = pq.read_table(path, columns=['audio']).to_pylist()

def get_audio_bytes(record):
    cell = parquet_cache[record['parquet_file']][int(record['row_index'])]['audio']
    audio_bytes = cell.get('bytes') if isinstance(cell, dict) else None
    if not audio_bytes:
        raise ValueError(f"audio bytes 없음: {record['sample_id']}")
    return audio_bytes

print('JSONL:', len(records))
print('parquet:', {k: len(v) for k, v in parquet_cache.items()})
print(json.dumps(records[0], ensure_ascii=False, indent=2)[:1200])

JSONL: 777
parquet: {'test-00000-of-00001.parquet': 777}
{
  "sample_id": "test_000000",
  "source": "fleurs",
  "transcription_my": "အော့ဆလော့ တောကြောင်အဝါသည် ကြွက်များကို စားရန် (သားကောင်) အတွက် မြေပြင်ပေါ်တွင် ၎င်းတို့သွားရောက်ခဲ့သည့်နေရာအား အနံ့ခံခြင်းဖြင့် အနံ့မှတဆင့် နောက်ယောင်ခံ၍ ရှာဖွေကြောင်း သိပ္ပံပညာရှင်များက ထင်မြင်ကြသည်။",
  "duration_seconds": 21.6,
  "parquet_file": "test-00000-of-00001.parquet",
  "row_index": 0,
  "translation_en": "Scientists think that ocelots follow and find animals to eat (prey) by smell, sniffing for where they've been on the ground.",
  "translation_status": "approved_reference",
  "translation_method": "fleurs_parallel_reference",
  "translation_note": ""
}


In [14]:
# 모델 로드: 공식 Gemma 4 audio 예제의 any-to-any pipeline 사용
from transformers import pipeline

pipe = pipeline(
    task='any-to-any',
    model=MODEL_ID,
    device_map='auto',
    dtype='auto',
)
print('loaded:', MODEL_ID)

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

loaded: google/gemma-4-E2B-it


In [15]:
ASR_PROMPT = '''Transcribe the following speech segment in Burmese into Burmese text.
Output only the Burmese transcription, with no translation, explanation, labels, or newlines.
Preserve names and numbers faithfully.'''.strip()

S2TT_PROMPT = '''Translate the following Burmese speech faithfully into English.
Output only the English translation, with no Burmese transcription, explanation, labels, or commentary.
Do not summarize. Preserve all names, numbers, quotations, and factual details.'''.strip()

def clean_generation(value):
    if isinstance(value, list):
        # Transformers 버전에 따라 generated_text가 content block list일 수 있음
        pieces = []
        for item in value:
            if isinstance(item, dict) and item.get('text'):
                pieces.append(str(item['text']))
            elif isinstance(item, str):
                pieces.append(item)
        value = ' '.join(pieces)
    text = str(value or '')
    text = text.replace('<turn|>', '').replace('<|end_of_turn|>', '').strip()
    text = re.sub(r'^(English|Burmese|Myanmar|Translation|Transcript)\s*:\s*', '', text, flags=re.I)
    return ' '.join(text.split())

def decode_audio(audio_bytes):
    waveform, sr = sf.read(io.BytesIO(audio_bytes), dtype='float32', always_2d=False)
    if waveform.ndim == 2:
        waveform = waveform.mean(axis=1)
    if sr != 16000:
        # scipy 없이 선형 보간. 현재 데이터는 전부 16 kHz라 정상적으로는 실행되지 않음.
        old_x = np.linspace(0.0, 1.0, len(waveform), endpoint=False)
        new_len = round(len(waveform) * 16000 / sr)
        new_x = np.linspace(0.0, 1.0, new_len, endpoint=False)
        waveform = np.interp(new_x, old_x, waveform).astype(np.float32)
        sr = 16000
    return np.asarray(waveform, dtype=np.float32), sr

def split_at_low_energy(waveform, sr=16000, max_seconds=29.0):
    max_samples = int(max_seconds * sr)
    if len(waveform) <= max_samples:
        return [waveform]
    chunks, start = [], 0
    search = int(2.0 * sr)
    frame = max(1, int(0.04 * sr))
    while len(waveform) - start > max_samples:
        target = start + max_samples
        lo = max(start + int(5 * sr), target - search)
        candidates = range(lo, target, frame)
        cut = min(candidates, key=lambda p: float(np.mean(waveform[p:p+frame] ** 2)))
        chunks.append(waveform[start:cut])
        start = cut
    if start < len(waveform):
        chunks.append(waveform[start:])
    return chunks

def infer_one_chunk(chunk, prompt, max_new_tokens):
    tmp_name = None
    try:
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
            tmp_name = tmp.name
        sf.write(tmp_name, chunk, 16000, subtype='PCM_16')
        messages = [{'role': 'user', 'content': [
            {'type': 'text', 'text': prompt},
            {'type': 'audio', 'audio': tmp_name},
        ]}]
        outputs = pipe(
            messages,
            return_full_text=False,
            generate_kwargs={'max_new_tokens': max_new_tokens, 'do_sample': False},
        )
        return clean_generation(outputs[0]['generated_text'])
    finally:
        if tmp_name and os.path.exists(tmp_name):
            os.unlink(tmp_name)

def infer_audio(audio_bytes, prompt, max_new_tokens):
    waveform, sr = decode_audio(audio_bytes)
    chunks = split_at_low_energy(waveform, sr, MAX_AUDIO_SECONDS)
    texts = [infer_one_chunk(c, prompt, max_new_tokens) for c in chunks]
    return ' '.join(t for t in texts if t).strip(), len(chunks), len(waveform) / sr


In [16]:
# 첫 샘플 단독 확인
sample = records[0]
audio_bytes = get_audio_bytes(sample)
asr_text, n_chunks, actual_duration = infer_audio(audio_bytes, ASR_PROMPT, MAX_NEW_TOKENS_ASR)
s2tt_text, _, _ = infer_audio(audio_bytes, S2TT_PROMPT, MAX_NEW_TOKENS_S2TT)
print('sample_id :', sample['sample_id'])
print('duration  :', actual_duration, 'chunks:', n_chunks)
print('Gold MY   :', sample['transcription_my'])
print('Gemma ASR :', asr_text)
print('Gold EN   :', sample['translation_en'])
print('Gemma S2TT:', s2tt_text)

sample_id : test_000000
duration  : 21.6 chunks: 1
Gold MY   : အော့ဆလော့ တောကြောင်အဝါသည် ကြွက်များကို စားရန် (သားကောင်) အတွက် မြေပြင်ပေါ်တွင် ၎င်းတို့သွားရောက်ခဲ့သည့်နေရာအား အနံ့ခံခြင်းဖြင့် အနံ့မှတဆင့် နောက်ယောင်ခံ၍ ရှာဖွေကြောင်း သိပ္ပံပညာရှင်များက ထင်မြင်ကြသည်။
Gemma ASR : အစလိုသောကြောင့် အဝတီ တွက်များကိုစားရန် သောက်ကောင်အတွက် မြေပြင်ပေါ်တွင်၎င်းတို့ သွားရောက်ခဲ့သည့်နေရာအနံ့ခံခြင်းဖြင့် အနံ့မသ슨နောက်ရောင်ခံ၍ ရှောင်ဖယ်ကြောင်းသိပ္ပံပညာရှင်များက တွေ့မြင်ကြသည်။
Gold EN   : Scientists think that ocelots follow and find animals to eat (prey) by smell, sniffing for where they've been on the ground.
Gemma S2TT: It was a matter that was widely discussed among the elderly, young people, and the general public, and during the period when the nation was facing economic hardship, various political parties were emerging.


In [17]:
# 중단 후 재개 가능한 전체 생성 루프
def load_latest_results(path):
    latest = {}
    if path.exists():
        with path.open('r', encoding='utf-8') as f:
            for line in f:
                try:
                    row = json.loads(line)
                    latest[row['sample_id']] = row
                except Exception:
                    pass
    return latest

existing = load_latest_results(PREDICTIONS_PATH)
targets = records if LIMIT is None else records[:LIMIT]
print(f'targets={len(targets)}, existing={len(existing)}')

with PREDICTIONS_PATH.open('a', encoding='utf-8') as out_f:
    for record in tqdm(targets):
        sid = record['sample_id']
        old = existing.get(sid)
        if old and old.get('generation_status') == 'success' and old.get('model_id') == MODEL_ID:
            continue

        result = {
            'sample_id': sid,
            'source': record.get('source'),
            'translation_status': record.get('translation_status'),
            'transcription_my': record.get('transcription_my', ''),
            'translation_en': record.get('translation_en', ''),
            'duration_seconds': record.get('duration_seconds'),
            'model_id': MODEL_ID,
        }
        try:
            audio_bytes = get_audio_bytes(record)
            waveform, sr = decode_audio(audio_bytes)
            chunks = split_at_low_energy(waveform, sr, MAX_AUDIO_SECONDS)
            result['actual_duration_seconds'] = len(waveform) / sr
            result['num_chunks'] = len(chunks)
            if RUN_ASR:
                result['gemma_asr_my'] = ' '.join(
                    infer_one_chunk(c, ASR_PROMPT, MAX_NEW_TOKENS_ASR) for c in chunks
                ).strip()
            if RUN_S2TT:
                result['gemma_s2tt_en'] = ' '.join(
                    infer_one_chunk(c, S2TT_PROMPT, MAX_NEW_TOKENS_S2TT) for c in chunks
                ).strip()
            result['generation_status'] = 'success'
            result['generation_error'] = ''
        except Exception as exc:
            result['generation_status'] = 'error'
            result['generation_error'] = repr(exc)

        out_f.write(json.dumps(result, ensure_ascii=False) + '\n')
        out_f.flush()
        existing[sid] = result

print('완료:', PREDICTIONS_PATH)

targets=777, existing=5


  0%|          | 0/777 [00:00<?, ?it/s]

완료: /content/drive/MyDrive/S2ST/baselines/gemma-4-E2B-it_mya_eng_s2tt/predictions.jsonl


## Smoke test 확인 후 전체 실행

위 5개 결과가 정상적이면 설정 셀의 `LIMIT = None`으로 바꾼 뒤 **전체 생성 루프 셀만 다시 실행**하세요. 앞서 성공한 5개는 자동으로 건너뜁니다.

In [18]:
# 최신 결과만 모아 상태 확인
latest = load_latest_results(PREDICTIONS_PATH)
df = pd.DataFrame(latest.values())
display(df.groupby(['source', 'generation_status'], dropna=False).size().rename('count').to_frame())
display(df[df['generation_status'] == 'error'][['sample_id', 'source', 'generation_error']].head(20))
display(df[['sample_id', 'source', 'transcription_my', 'gemma_asr_my', 'translation_en', 'gemma_s2tt_en']].head(10))

,,count
source,generation_status,
fleurs,success,384
mig-burmese-audio-transcription,success,140
openslr,success,253


,sample_id,source,generation_error


,sample_id,source,transcription_my,gemma_asr_my,translation_en,gemma_s2tt_en
0,test_000000,fleurs,အော့ဆလော့ တောကြောင်အဝါသည် ကြွက်များကို စားရန် ...,အစလိုသောကြောင့် အဝတီ တွက်များကိုစားရန် သောက်ကေ...,Scientists think that ocelots follow and find ...,It was a matter that was widely discussed amon...
1,test_000001,fleurs,အလောင်းသည် ယင်းနေရာတွင် ပေါ်လာသည်မှာ တစ်ရက်ပင်...,အလောင်းတီ ရနညာတွင် ပေါ်လာသည်မှာ တရက်ပင်ရှိပြီဟ...,Police said that the body appeared to have bee...,"Although there are many people in Bolani, the ..."
2,test_000002,fleurs,"""ဘုရှ်၏ ပြောရေးဆိုခွင့်ရှိသူဂျောဒန် ဂျွန်ဒရိုး...",ပုရှီ ပြည်သူစုခွင်ရှိသူ ဂျော်ဒန် ဂျွန်ဒရိုကာ မ...,Bush spokesman Gordon Johndroe called North Ko...,"Bush E. Pei Soo Kwun-si-tu, the North Korean n..."
3,test_000003,fleurs,တိဘက်လူမျိုးတို့၏ ဗုဒ္ဓဘာသာသည် ဗုဒ္ဓ၏အဆုံးအမမျ...,ကြည်ပတ်တွေမြို့လို့ အိ ဗုဒ္ဓဘာသာနဲ့ ဗုဒ္ဓရဲ့ အ...,The Tibetan Buddhism is based on the teachings...,"The Buddha's teachings, the Buddha's profound ..."
4,test_000004,fleurs,များစွာသောကိစ္စရပ်များတွင် နိုင်ငံရပ်ခြားတွင် ...,ညှာသော ကြီးစားရများတွင် နိုင်ငံရေးချာတွင် နှိရ...,"In many cases, enrolling on a gap-year course ...",The government has approved the appointment of...
5,test_000005,fleurs,တင်ပြထားသည့် ရှုထောင့်များသည် အခြားနေရာတစ်ခုတွ...,တင်ရထား ဒီ ရှုတော် များ ဒီ အချားနေရာ တစ်ခု တွင...,"The views presented are often cursory, general...","When I was young, I didn't remember the days w..."
6,test_000006,fleurs,အစပိုင်းတွင် ဝတ်စုံကို အရှေ့ဘက်ရှိ ဘိုင်ဇန်တို...,အစပိုင်းတွင် ဝတ်ဆုံကို အရှေ့ဘက်ရှိ ဘိုင်းဆန်တိ...,At the beginning dress was heavily influenced ...,"At that time, the people of Wessong and Ashi P..."
7,test_000007,fleurs,ကျောင်းသားများသည် မကြာခဏဆိုသလို ဝေဖန်လေ့ရှိကြသ...,ကျောင်းသားများ ဒီမကြာခင်ဆိုတော့ ဝိပန်းလေးရှိja...,Since students are often the most critical aud...,"I don't know how to say it. When I was happy, ..."
8,test_000008,fleurs,ငါ့အတွက်တော့ ဒါက လုံးဝအဓိပ္ပါယ်မရှိဘူး၊ လုံးဝက...,ငါ အထောထတက လုံးဝ အထိတ်ပင်မရှိဘူး လုံးဝကို မတရာ...,That didn't seem to make sense to me; it certa...,"I don't have much respect, I don't know much."
9,test_000009,fleurs,"""""""စိတ်ဝင်စားဖွယ်ကောင်းသောရွာသို့ အေးအေးလူလူဖြ...",စိတ်ဝင်စားပဲ ကောင်းလို့ ရသွားသူ အင်းအီလူလူဖြင့...,It's worth half an hour to stroll about the in...,"The beautiful scenery, the gentle breeze, and ..."


In [19]:
# ASR CER 및 direct S2TT BLEU/chrF++ 평가
from jiwer import cer
from sacrebleu.metrics import BLEU, CHRF

bleu = BLEU(tokenize='13a', effective_order=True)
chrf = CHRF(word_order=2)

def normalize_my(text):
    text = unicodedata.normalize('NFKC', str(text or ''))
    text = ''.join(ch for ch in text if not unicodedata.category(ch).startswith(('P', 'Z')) and not ch.isspace())
    return text

def score_subset(frame, name):
    frame = frame[(frame['generation_status'] == 'success') & frame['translation_en'].fillna('').ne('')].copy()
    if frame.empty:
        return None
    refs = frame['translation_en'].astype(str).tolist()
    hyps = frame['gemma_s2tt_en'].fillna('').astype(str).tolist()
    asr_pairs = frame[frame['gemma_asr_my'].fillna('').ne('')]
    asr_cer = cer(
        [normalize_my(x) for x in asr_pairs['transcription_my']],
        [normalize_my(x) for x in asr_pairs['gemma_asr_my']],
    ) if len(asr_pairs) else np.nan
    return {
        'subset': name,
        'samples': len(frame),
        'asr_cer': asr_cer,
        'direct_s2tt_bleu': bleu.corpus_score(hyps, [refs]).score,
        'direct_s2tt_chrf++': chrf.corpus_score(hyps, [refs]).score,
        'chunked_samples': int((frame['num_chunks'].fillna(1) > 1).sum()),
    }

rows = []
all_scored = df[df['generation_status'] == 'success'].copy()
for name, part in [
    ('all_available', all_scored),
    ('approved_only', all_scored[all_scored['translation_status'] == 'approved_reference']),
]:
    row = score_subset(part, name)
    if row: rows.append(row)
for source, part in all_scored.groupby('source'):
    row = score_subset(part, f'source:{source}')
    if row: rows.append(row)

summary = pd.DataFrame(rows)
summary.to_csv(SUMMARY_PATH, index=False)
display(summary.round(3))
print('Predictions:', PREDICTIONS_PATH)
print('Summary    :', SUMMARY_PATH)

,subset,samples,asr_cer,direct_s2tt_bleu,direct_s2tt_chrf++,chunked_samples
0,all_available,777,0.322,1.308,18.166,9
1,approved_only,428,0.330,0.906,19.138,9
2,source:fleurs,384,0.328,0.927,19.482,9
3,source:mig-burmese-audio-transcription,140,0.449,0.636,12.147,0
4,source:openslr,253,0.249,2.473,16.785,0


Predictions: /content/drive/MyDrive/S2ST/baselines/gemma-4-E2B-it_mya_eng_s2tt/predictions.jsonl
Summary    : /content/drive/MyDrive/S2ST/baselines/gemma-4-E2B-it_mya_eng_s2tt/summary.csv
